# Chapter 10 — RAG Generation & Evaluation

*Where we are:* retrieval feeds **grounded generation**, and we learn to *measure* the answer.

```
ranked chunks →[ context builder → citation-aware prompt → LLM ]→ grounded answer → eval
```

We build the **context builder**, the **citation-aware prompt**, the **mock LLM provider**, the
**grounded-answer** function, and every **RAG metric** inline (all packaged in
`patentrag.generation` / `patentrag.evaluation`). No paid key is required.

In [1]:
# === Chapter 10 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 10 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 10 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 43. Retrieval ≠ context construction

Good context is **deduplicated**, **source-diverse**, **ordered**, and **within budget** — not just
top-k glued together. We implement the builder inline (packaged as
`patentrag.generation.ContextBuilder`).

In [2]:
import pandas as pd
from dataclasses import dataclass
from patentrag.models import Citation                    # the citation type (Ch 01)
from patentrag.dense import DenseRetriever               # dense retriever (Ch 07)
from patentrag.fusion import reciprocal_rank_fusion      # RRF (Ch 08)

chunks = bs.ensure("chunks"); by_id = {c.chunk_id: c for c in chunks}   # corpus chunks + lookup
bm25 = bs.ensure("bm25_index"); emb = bs.ensure("embeddings")            # retrievers
dense = DenseRetriever(emb["chunk_ids"], emb["matrix"])

query = "how are retrieval-aware embeddings used to search media objects?"
# hybrid candidate list: fuse BM25 + dense rankings, keep the top 20 chunks
fused = reciprocal_rank_fusion([[c for c, _ in bm25.search(query, 40)],
                                [c for c, _ in dense.search(query, 40)]])
ranked = [by_id[cid] for cid, _ in fused[:20]]           # the ranked chunks to build context from

@dataclass
class ContextPassage:                                    # one selected passage + its citation
    citation: Citation
    text: str
    tokens: int

def build_context(ranked_chunks, token_budget=1000, max_per_doc=2):
    """Select passages: dedup near-identical, cap per document (diversity), stop at the token budget."""
    used, per_doc, seen, out = 0, {}, set(), []          # running totals + trackers
    for c in ranked_chunks:                              # walk the ranking in order
        key = c.text[:120]                               # cheap near-duplicate key (first 120 chars)
        if key in seen:                                  # skip a passage we already included
            continue
        if per_doc.get(c.document_id, 0) >= max_per_doc: # enforce source diversity (<= max_per_doc/doc)
            continue
        if used + c.token_estimate > token_budget and out:  # stop once adding would exceed the budget
            break
        cit = Citation(document_id=c.document_id, publication_number=c.publication_number,
                       section=c.section, claim_number=c.claim_number, chunk_id=c.chunk_id, anchor=c.anchor)
        out.append(ContextPassage(cit, c.text, c.token_estimate))  # keep the passage
        seen.add(key); per_doc[c.document_id] = per_doc.get(c.document_id, 0) + 1; used += c.token_estimate
    return out

passages = build_context(ranked, token_budget=1000, max_per_doc=2)
print(f"from {len(ranked)} ranked chunks → {len(passages)} context passages (deduped, ≤2/doc, ≤1000 tok)")
pd.DataFrame([{"citation": p.citation.label()[:50], "tokens": p.tokens, "text": p.text[:40]} for p in passages])

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10090.70it/s]

from 20 ranked chunks → 2 context passages (deduped, ≤2/doc, ≤1000 tok)


,citation,tokens,text
0,[PATENT=US11971885B2 | SECTION=Image Search Ap...,99,The retrieval network 110 can efficientl
1,[PATENT=US11971885B2 | SECTION=Image Search Ap...,50,The database 120 stores the media object


## 44. Citation-aware prompt construction

Each passage carries a **stable citation tag** `[PATENT=… | SECTION/CLAIM=… | CHUNK=…]`, and the
system prompt requires the model to answer **with** those tags — making the output mechanically
verifiable downstream.

In [3]:
SYSTEM_PROMPT = (                                        # instructions given to the model as the system role
    "You are a patent analysis assistant. Answer the question using ONLY the context passages. "
    "Every statement must cite the passage it comes from using that passage's bracket tag verbatim, "
    "e.g. [PATENT=US1234567B2 | CLAIM=1 | CHUNK=abc123]. If the context lacks the answer, say so.")

def build_prompt(query, passages):                       # assemble the user prompt from the context
    # each block = the citation tag on its own line, then the passage text
    blocks = [f"{p.citation.label()}\n{p.text}" for p in passages]
    return "Context:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {query}\nAnswer:"

prompt = build_prompt(query, passages)
print("SYSTEM:", SYSTEM_PROMPT[:120], "...\n")
print("PROMPT (head):\n", prompt[:380], "...")

SYSTEM: You are a patent analysis assistant. Answer the question using ONLY the context passages. Every statement must cite the  ...

PROMPT (head):
 Context:

[PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=6acd8fe0d9a8]
The retrieval network 110 can efficiently and accurately compare a query object and a set of media objects in a common and sparse embedding space, and scale to millions of contents. As a result, the system can be further applied to multiple embeddings. For example, the query may include mult ...


## 45. Grounded generation — the `LLMProvider` + a deterministic mock

The **default provider is a deterministic mock** that answers only from the supplied passages,
echoing their tags — grounded *by construction*, so the pipeline runs offline. (A real
OpenAI-compatible endpoint is an optional env-gated path.) We inline the mock and the
`generate_answer` flow (packaged in `patentrag.generation`).

In [4]:
import re
from patentrag.models import PatentChunk                 # (unused directly; imported for parity/clarity)

_LABEL_RE = re.compile(r"\[PATENT=[^\]]+\]")             # matches a whole citation tag
# matches (tag, passage-text) pairs inside the prompt's Context block:
_PASSAGE_RE = re.compile(r"(\[PATENT=[^\]]+\])\n(.*?)(?=\n\[PATENT=|\nQuestion:|\Z)", re.DOTALL)

def mock_complete(prompt, max_passages=2):
    """Deterministic 'LLM': extract the first passages from the prompt and echo their lead sentence + tag."""
    found = _PASSAGE_RE.findall(prompt)                  # [(tag, text), ...] parsed straight from the prompt
    if not found:
        return "I could not find supporting passages in the provided context."
    parts = []
    for label, text in found[:max_passages]:            # use the top one or two passages
        first = re.split(r"(?<=[.!?])\s+", text.strip())[0]   # the passage's first sentence
        parts.append(f"{first} {label}")               # sentence followed by its citation tag
    return " ".join(parts)

def extract_citations(answer_text, passages):            # map the tags the 'model' emitted back to citations
    by_chunk = {p.citation.chunk_id: p.citation for p in passages}   # chunk_id -> Citation
    cites = []
    for tag in _LABEL_RE.findall(answer_text):          # every tag in the answer
        m = re.search(r"CHUNK=([A-Za-z0-9]+)", tag)     # pull the chunk id out of the tag
        if m and m.group(1) in by_chunk and by_chunk[m.group(1)] not in cites:
            cites.append(by_chunk[m.group(1)])          # keep the matching citation (dedup)
    return cites

answer_text = mock_complete(prompt)                      # generate (offline, deterministic)
citations = extract_citations(answer_text, passages)     # parse citations back out
print("ANSWER:\n", answer_text[:360])
print("\ncitations:", [c.label() for c in citations])
retrieved_ids = {c.chunk_id for c in ranked}             # everything that was retrieved
print("all citations came from retrieved context:", all(c.chunk_id in retrieved_ids for c in citations))

ANSWER:
 The retrieval network 110 can efficiently and accurately compare a query object and a set of media objects in a common and sparse embedding space, and scale to millions of contents. [PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=6acd8fe0d9a8] The database 120 stores the media objects and the respective sparse embeddings for each of the media

citations: ['[PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=6acd8fe0d9a8]', '[PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=09c4ab951e4f]']
all citations came from retrieved context: True


> This is `patentrag.generation.MockLLMProvider` + `generate_answer`. The packaged `generate_answer`
> wraps context-building → prompt → provider → citation extraction into one call, and picks a live
> provider automatically if `LLM_API_KEY`/`LLM_BASE_URL` are set.

In [5]:
from patentrag.generation import generate_answer, MockLLMProvider   # the packaged flow
ans = generate_answer(query, ranked, provider=MockLLMProvider())     # same result, one call
print("packaged answer citations:", len(ans.citations), "| grounded:",
      all(c.chunk_id in retrieved_ids for c in ans.citations))

packaged answer citations: 2 | grounded: True


## 46–47. Why LLM eval is hard; deterministic checks first

Generation is **nondeterministic**, answers can be **correct in many wordings**, and there's often
**no single reference**. Judges help but are biased (**position**, **verbosity**, **self-preference**).
So we start with **deterministic** checks: citation validity + structured-schema validity.

In [6]:
# citation validity — deterministic (does each cited chunk exist and was it retrieved?):
def verify_citation(cit, retrieved, by_id):
    exists = cit.chunk_id in by_id                       # the chunk is a real corpus chunk
    was_retrieved = cit.chunk_id in retrieved            # ...and it was actually retrieved for this query
    offsets_ok = cit.anchor is not None and cit.anchor.norm_start is not None  # its offsets resolve
    return {"label": cit.label()[:44], "exists": exists, "retrieved": was_retrieved, "valid": exists and was_retrieved and offsets_ok}

pd.DataFrame([verify_citation(c, retrieved_ids, by_id) for c in ans.citations])

,label,exists,retrieved,valid
0,[PATENT=US11971885B2 | SECTION=Image Search,True,True,True
1,[PATENT=US11971885B2 | SECTION=Image Search,True,True,True


## 48. Semantic evaluation — answer relevance & faithfulness

Beyond exact checks: is the answer **on-topic** (answer relevance), and is every sentence
**supported by the context** (faithfulness)? Both via embeddings. Inline (packaged in
`patentrag.evaluation`).

In [7]:
import numpy as np
from patentrag.dense import embed                        # the same embedder from Ch 07
def _cos(a, b):                                          # cosine between two vectors
    return float(a @ b / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12))

def answer_relevance(question, answer):
    """Cosine similarity of the question and answer embeddings (is the answer about the question?)."""
    v = embed([question, answer])                        # embed both at once
    return max(0.0, _cos(v[0], v[1]))

def faithfulness(answer, contexts, threshold=0.55):
    """Fraction of answer sentences that are close to SOME context sentence (a groundedness proxy)."""
    sents = [s for s in re.split(r"(?<=[.!?])\s+", answer.strip()) if s]   # answer -> sentences
    ctx = [s for c in contexts for s in re.split(r"(?<=[.!?])\s+", c) if s] or contexts  # context -> sentences
    if not sents or not ctx:
        return 0.0
    A, C = embed(sents), embed(ctx)                      # embed both sets
    sims = A @ C.T                                        # (n_answer, n_ctx) similarity matrix
    return float((sims.max(axis=1) >= threshold).sum()) / len(sents)  # each answer sent's best match >= thresh?

clean_answer = _LABEL_RE.sub("", ans.answer).strip()     # drop citation tags before scoring the prose
ctx_texts = [by_id[c.chunk_id].text for c in ans.citations]
print(f"answer relevance (question ↔ answer): {answer_relevance(query, clean_answer):.2f}")
print(f"faithfulness (answer ⊑ context)     : {faithfulness(clean_answer, ctx_texts):.2f}")

answer relevance (question ↔ answer): 0.68
faithfulness (answer ⊑ context)     : 1.00


## 49–51. LLM-as-judge; decompose RAG quality; core metrics

RAG quality decomposes: **retriever → context → generator → answer**. A wrong answer does not
automatically implicate the LLM. **Context precision/recall** measure the retriever; **faithfulness**
the generator; **answer relevance** the final fit. An LLM-as-judge (env-gated) scores a rubric; keep
deterministic checks alongside it.

In [8]:
def context_precision(retrieved_doc_ids, relevant):      # of retrieved docs, how many are relevant?
    return sum(1 for x in retrieved_doc_ids if x in relevant) / len(retrieved_doc_ids) if retrieved_doc_ids else 0.0
def context_recall(retrieved_doc_ids, relevant):         # of relevant docs, how many were retrieved?
    return len(set(retrieved_doc_ids) & relevant) / len(relevant) if relevant else 1.0

# evaluate on one labelled benchmark example (Ch 09 dataset)
ex = next(e for e in bs.ensure("eval_dataset") if "media objects" in e.query)
def ranked_docs(chunk_ids):                              # collapse a chunk ranking to unique doc ranking
    seen = []
    for cid in chunk_ids:
        did = by_id[cid].document_id
        if did not in seen:
            seen.append(did)
    return seen
r_docs = ranked_docs([c for c, _ in fused[:10]])
ans2 = generate_answer(ex.query, ranked, provider=MockLLMProvider())
pd.DataFrame([{
    "context_precision@5": round(context_precision(r_docs[:5], ex.relevant_doc_ids), 2),
    "context_recall@5":    round(context_recall(r_docs[:5], ex.relevant_doc_ids), 2),
    "faithfulness":        round(faithfulness(_LABEL_RE.sub('', ans2.answer), [by_id[c.chunk_id].text for c in ans2.citations]), 2),
    "answer_relevance":    round(answer_relevance(ex.query, _LABEL_RE.sub('', ans2.answer)), 2),
}])

,context_precision@5,context_recall@5,faithfulness,answer_relevance
0,1.0,1.0,1.0,0.51


## 52. Frameworks (Ragas / DeepEval)

The metrics above are what **Ragas** and **DeepEval** compute. Their live metrics use an LLM judge
+ embeddings, so we implemented them manually to stay offline; enable a framework by configuring an
LLM key.

In [9]:
from patentrag.generation import OpenAICompatibleProvider
if OpenAICompatibleProvider.available():
    print("A live endpoint is configured — a Ragas/DeepEval run could be wired here.")
else:
    print("SKIPPED (loudly): Ragas/DeepEval faithfulness/answer-relevance use an LLM judge; with no")
    print("LLM_API_KEY we compute the SAME metrics manually above so the chapter runs offline.")

SKIPPED (loudly): Ragas/DeepEval faithfulness/answer-relevance use an LLM judge; with no
LLM_API_KEY we compute the SAME metrics manually above so the chapter runs offline.


## 53. RAG failure attribution (A–E)

Different failures need different metrics to detect. We stage five *illustrative* scenarios and
show which signal flags each.

In [10]:
gold_ctx = "Retrieval-aware embedding trains dense embeddings for the retrieval objective."
scenarios = [
    ("A: retriever failed", context_recall([], {"US11971885B2"}), "context_recall ↓"),
    ("B: right evidence buried", context_recall(["Xdoc", "US11971885B2"], {"US11971885B2"}), "present but low rank → MRR ↓"),
    ("C: LLM hallucinated", round(faithfulness("The system uses quantum entanglement.", [gold_ctx]), 2), "faithfulness ↓"),
    ("D: wrong citation", "citation.valid = False", "citation verification ↓"),
    ("E: grounded but incomplete", round(faithfulness("Embeddings are dense.", [gold_ctx]), 2), "faithful but low answer_relevance"),
]
pd.DataFrame([{"failure": s, "signal": v, "detected_by": d} for s, v, d in scenarios])

,failure,signal,detected_by
0,A: retriever failed,0.0,context_recall ↓
1,B: right evidence buried,1.0,present but low rank → MRR ↓
2,C: LLM hallucinated,0.0,faithfulness ↓
3,D: wrong citation,citation.valid = False,citation verification ↓
4,E: grounded but incomplete,0.0,faithful but low answer_relevance


> Inline `build_context`, `build_prompt`, `mock_complete`/`extract_citations`, `answer_relevance`,
> `faithfulness`, `context_precision/recall` above == `patentrag.generation` + `patentrag.evaluation`.

**Production implications.** Log the decomposed metrics per request; a *faithfulness* alert means
fix generation/prompting, a *context recall* alert means fix retrieval. Keep a deterministic layer
(citation + schema validity) that can hard-fail a response regardless of a judge.

## Chapter invariants

In [11]:
assert ans.citations and all(c.chunk_id in retrieved_ids for c in ans.citations)   # grounded citations
assert 0.0 <= answer_relevance(query, clean_answer) <= 1.0                          # metric bounds
assert faithfulness("Embeddings are dense.", [gold_ctx]) >= 0.0
assert context_recall(["US11971885B2"], {"US11971885B2"}) == 1.0                   # recall correctness
# inline mock matches packaged provider (same lead-sentence + tag echo):
assert extract_citations(mock_complete(prompt), passages)                          # inline mock cites
print("All Chapter 10 invariants hold.")

All Chapter 10 invariants hold.


In [12]:
# === Chapter 10 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['sentence-transformers', 'pydantic', 'pandas']
print("Chapter 10 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 10 VALIDATION: PASS")

Chapter 10 — environment
  Python : 3.12.10 on Windows 11
  sentence-transformers   : 6.0.0
  pydantic                : 2.13.3
  pandas                  : 3.0.2

CHAPTER 10 VALIDATION: PASS
